# IOAI — 2025 Lab Lab6 Defanging Lion (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!pip -q install kornia
print('사자 이미지·somenetwork.py·인코더/디코더 가중치는 노트북 셀이 공개 GCS 에서 자동 다운로드합니다(무거움).')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Defanging a lion

## Problem statement

In your safari trip, you came across a dangerous lion.

In [ ]:
!curl https://storage.googleapis.com/aiolympiadmy/ioai-2025-tsp/ioai2025_tsp_selection2/defanging_lion/francesco-ZxNKxnR32Ng-unsplash.jpg -o lion.jpg

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


Ensure your safety by transforming the lion in this picture into a slightly less dangerous animal: an african buffalo.

Here is how your work will be scored:

- 1 pt awarded for demonstrating that your transformed lion is closer in meaning to the text "a peaceful african buffalo" than the text "a prowling lion on the hunt" by `openai/clip-vit-base-patch32`

> For CLIP scoring, use cosine similarity for your calculation. Scoring less than 0.5 using the reference implementation of binary softmax is sufficient to be considered as closer to buffalo. To ensure that your implementation is correct, and to save you the headache of losing progress due to not realizing you need to normalize your vectors, sample code is provided below as a reference. I am not mandating you use this code because you might prefer to handle the vectors directly instead of being wrapped in transformers as a dictionary with key "pixel_values"

- 1 pt awarded if the point above is earned, and your transformed lion has less than 0.3 SSIM loss compared to the original lion.

> For SSIM loss, install `kornia` and look at the sample code provided below.

- 1 pt awarded if `openai/clip-vit-base-patch32` considers the softmaxed probability that your transformed lion is a buffalo is higher than 0.7 (i.e. lion < 0.3) AND SSIM loss is less than 0.3
- 2 pts awarded for demonstrating that your transformed lion is so convincingly transformed into a buffalo, that it visually looks more like a buffalo than a lion. SSIM loss criterion not applicable for these points.
    - If it clearly looks like a buffalo to me, these pts will count. If not, I will load any readily-available image classification model pretrained on ImageNet and check if the transformed image is closer to class 291 (lion) or class 346 (water buffalo). The reason for this evaluation is to make sure that you aren't creating an adversarial image that is barely modified but scores the correct class.
    - Make sure that your image is shown in your notebook for further inspection.
- 1 pt awarded for explaining your thought process and reasoning for your work done. Keep it brief, one short paragraph is enough!

Partial credit will be granted at discretion.

You are allowed to use:

- any pretrained model from `torchvision` and `transformers`
- any code and pretrained models made available to you as part of this problem
- standard ML libraries (e.g. numpy, sklearn, albumentations, einops)
- additional images of your own

You are prohibited to use:

- pretrained models that are not from the allowed sources. Off the shelf models, dedicated libraries and github repos are not allowed.
- responses from LLM APIs like OpenAI and using the response as your solution
- hosted local LLMs to generate images and use the generated image as your solution
- any technique that amounts to replacing the lion with a different picture

You might find this mysterious SomeNetwork useful.

Requirements: 
- Install `attrs`. You should already have `numpy`, `torch`, and `torchvision`
- Download `somenetwork.py` from https://storage.googleapis.com/aiolympiadmy/ioai-2025-tsp/ioai2025_tsp_selection2/defanging_lion/somenetwork.py
- Download weights (370+ MB combined):
    - `encoder.pt` from https://storage.googleapis.com/aiolympiadmy/ioai-2025-tsp/ioai2025_tsp_selection2/defanging_lion/encoder.pt
    - `decoder.pt` from https://storage.googleapis.com/aiolympiadmy/ioai-2025-tsp/ioai2025_tsp_selection2/defanging_lion/decoder.pt
- Download `using-somenetwork.ipynb` from https://storage.googleapis.com/aiolympiadmy/ioai-2025-tsp/ioai2025_tsp_selection2/defanging_lion/using-somenetwork.ipynb to see how to use it

## Sample code to help you get started

```python
device = "cuda" if torch.cuda.is_available() else "cpu"

from PIL import Image
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

text_inputs = processor(
    text="a peaceful african buffalo", return_tensors="pt", padding=True
).to(device)
buffalo_text_embeddings = model.get_text_features(**text_inputs).float()
buffalo_text_embeddings = F.normalize(buffalo_text_embeddings, p=2, dim=-1)

text_inputs = processor(text="a prowling lion on the hunt", return_tensors="pt", padding=True).to(
    device
)
lion_text_embeddings = model.get_text_features(**text_inputs).float()
lion_text_embeddings = F.normalize(lion_text_embeddings, p=2, dim=-1)

image = Image.open("lion.jpg").convert("RGB")
inputs = processor(
    images=image,
    return_tensors="pt",
    padding=True,
    do_rescale=True,
    do_normalize=True,
)
inputs = {k: v.to(device) for k, v in inputs.items()}

def compute_clip_lion2buffalo_loss(image_inputs):
    image_embeddings = model.get_image_features(**image_inputs).float()
    image_embeddings = F.normalize(image_embeddings, p=2, dim=-1)

    global lion_text_embeddings
    global buffalo_text_embeddings

    # Cosine similarity (higher = more similar)
    lion_sim = (image_embeddings @ lion_text_embeddings.T).squeeze()
    buffalo_sim = (image_embeddings @ buffalo_text_embeddings.T).squeeze()

    scores = torch.stack([lion_sim, buffalo_sim])
    probs = torch.softmax(scores, dim=0)
    lion_score = probs[0]
    
    # <0.5 means closer to buffalo
    return lion_score
```

```python
from kornia.losses import SSIMLoss

ssim_loss_fxn = SSIMLoss(window_size=7).to(device)

a = torch.rand(1, 3, 256, 256)
b = torch.rand(1, 3, 256, 256)

ssim_loss_fxn(a, b)
```

## Your work below

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


---

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


## Demo

This section makes sure everything works fine.

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


## Main

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


Align the two animals carefully to achieve the best training outcome!

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


## Explanation

We implemented a latent space mixing approach to transform a lion image into a buffalo. Through gradient-based optimization, we learned a binary mask that selectively replaces parts of the lion's latent code with corresponding regions from the buffalo's latent code. The optimization objective combines perceptual loss (using ResNet features) and SSIM loss to maintain visual quality while encouraging buffalo-like characteristics.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv', 'submission.zip', 'submission.jsonl', 'submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)